# Proyecto Integrador: Predicción de Enfermedades Cardíacas (MLOps Local)


**Autores:** Nátaly Cárdenas and Laura Rivera

Este cuaderno integra el flujo completo del proyecto:
1. **EDA**: Análisis Exploratorio de Datos con informe visual automático (YData Profiling).
2. **Modelado**: Pipeline de preprocesamiento, GridSearchCV y evaluación de 4 modelos.
3. **Resultados**: Comparación de métricas y visualización de la curva ROC del mejor modelo.
4. **Exportación**: Generación del artefacto `model.joblib` para ser consumido por Docker/Kubernetes.

In [1]:
# 1. Importación de librerías
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

print("Librerías importadas correctamente.")

Librerías importadas correctamente.


## 2. Carga de Datos y Análisis Exploratorio (EDA)

In [2]:
# Cargar datos
filename = r"C:\Users\laure\Documents\Proyectos\Heart_Predition\heart.csv"
df = pd.read_csv(filename)
print(" Dataset cargado correctamente.")
display(df.head())

 Dataset cargado correctamente.


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


### 2.1 Informe Visual Automático (YData Profiling)
Se genera un informe HTML interactivo para analizar estadísticas, correlaciones y valores atípicos.

In [3]:
from ydata_profiling import ProfileReport

# Generamos el reporte
profile = ProfileReport(df, title="Informe de los datos", explorative=True)
profile.to_notebook_iframe()
print("Informe visual generado: informe_datos_heart.html")

ModuleNotFoundError: No module named 'ydata_profiling'

### **Resultados del EDA**
- El dataset contiene **918 observaciones** y **12 variables**.
- No se encontraron valores nulos ni filas duplicadas.
- Se detectaron **5 variables numéricas** y **6 variables categóricas**.
- **Correlación alta detectada:** Las variables `ChestPainType` y `ST_Slope` están altamente correlacionadas con la variable objetivo `HeartDisease`.
- **Valores atípicos:** Se detectaron 172 valores `0` en la variable `Cholesterol` (18.7%) y 368 valores `0` en `Oldpeak` (40.1%), lo cual es relevante para el preprocesamiento.

## 3. Preparación de Datos y Pipeline de Modelado

Se define un Pipeline con un escalador `MinMaxScaler` y un clasificador. Se utiliza **GridSearchCV** con validación cruzada (5 folds) y `roc_auc` como métrica de scoring.

In [19]:
# Separar variables
X = df.drop("HeartDisease", axis=1)
y = df["HeartDisease"]

# División 80/20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Tamaño de entrenamiento: {X_train.shape}")
print(f"Tamaño de prueba: {X_test.shape}")

Tamaño de entrenamiento: (734, 11)
Tamaño de prueba: (184, 11)


In [24]:
# Función reutilizable para entrenar
def train_pipeline(X_train, y_train, model, param_grid):
    pipe = Pipeline([("scaler", MinMaxScaler()), ("clf", model)])
    # error_score='raise' nos ayudará a ver errores específicos si algo falla
    grid = GridSearchCV(pipe, param_grid, cv=5, scoring="roc_auc", error_score='raise')
    grid.fit(X_train, y_train)
    return grid

models = {
    "SVC": (SVC(probability=True), {"clf__C": [0.1, 1, 10], "clf__gamma": [0.01, 0.1, 1]}),
    "RandomForest": (RandomForestClassifier(random_state=42), {"clf__n_estimators": [50, 100], "clf__max_depth": [None, 10]}),
    "KNN": (KNeighborsClassifier(), {"clf__n_neighbors": [3, 5, 7]}),
    "LogisticRegression": (LogisticRegression(max_iter=1000), {"clf__C": [0.1, 1, 10]})
}

results = {}
for name, (model, params) in models.items():
    print(f"Entrenando {name}...")
    grid = train_pipeline(X_train, y_train, model, params)
    results[name] = grid
    print(f"Mejores parámetros para {name}: {grid.best_params_}")

Entrenando SVC...


ValueError: could not convert string to float: 'M'

## 4. Evaluación de Modelos y Resultados

Se calculan las métricas de **Accuracy** y **AUC** sobre el conjunto de prueba para cada modelo.

In [ ]:
# Evaluar en el set de prueba
summary = []
for name, grid in results.items():
    y_pred = grid.predict(X_test)
    y_prob = grid.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    
    summary.append({"Modelo": name, "Accuracy": acc, "AUC": auc})

# Tabla de resultados
df_results = pd.DataFrame(summary).sort_values(by="AUC", ascending=False)
print("🏆 Ranking de modelos evaluados:")
display(df_results)

### **Resultados de los Modelos**

| Modelo               |   Accuracy |   AUC   |
|:---------------------|:----------:|:-------:|
| **SVC**              |   0.891    |  0.941  |
| **RandomForest**     |   0.859    |  0.921  |
| **LogisticRegression**|   0.870    |  0.911  |
| **KNN**              |   0.848    |  0.906  |

**Análisis:** El modelo **Support Vector Classifier (SVC)** demostró el mejor desempeño, alcanzando un AUC de **0.94**, lo que indica una excelente capacidad para distinguir entre pacientes con y sin riesgo de falla cardíaca.

In [23]:
# Visualización de la Curva ROC del mejor modelo (SVC)
best_model_name = "SVC"  # Seleccionamos el ganador según el ranking
best_grid = results[best_model_name]

y_prob = best_grid.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC SVC (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate (FPR)")
plt.ylabel("True Positive Rate (TPR)")
plt.title(f"Curva ROC - Mejor Modelo ({best_model_name})")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

KeyError: 'SVC'

## 5. Exportación del Modelo Final

Se guarda el mejor modelo (SVC) en un archivo `.joblib`. Este archivo será utilizado por la API en `app/api.py`.

In [ ]:
# Crear carpeta app si no existe
if not os.path.exists("app"):
    os.makedirs("app")

# Guardar el mejor modelo
joblib.dump(best_grid.best_estimator_, "app/model.joblib")
print(f"✅ Modelo guardado en 'app/model.joblib'. Modelo elegido: {best_model_name}")

## 6. Estructura del Despliegue (Docker y API)

Este notebook se encarga de la parte de datos y modelos. Para completar el despliegue local, se requieren los siguientes archivos en la carpeta del proyecto:

**`docker/requirements.txt`**
```text
fastapi
uvicorn
scikit-learn
joblib
pydantic
pandas
numpy
```

**`app/api.py`** (El cual cargará el modelo entrenado)
```python
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import numpy as np

model = joblib.load("app/model.joblib")
app = FastAPI()

class Input(BaseModel):
    features: list

@app.post("/predict")
def predict(data: Input):
    X = np.array(data.features).reshape(1, -1)
    proba = model.predict_proba(X)[0][1]
    return {"heart_disease_probability": proba, "prediction": int(proba > 0.5)}
```

**`docker/Dockerfile`**
```dockerfile
FROM python:3.10-slim
WORKDIR /app
COPY docker/requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app/ /app/
CMD ["uvicorn", "app.api:app", "--host", "0.0.0.0", "--port", "8000"]
```

## 7. Conclusión

- El análisis EDA permitió identificar que el dataset está balanceado y sin datos faltantes, y que la variable `ChestPainType` es clave para el diagnóstico.
- Se compararon 4 algoritmos de clasificación. El modelo **SVC** obtuvo el mejor rendimiento con un **AUC de 0.941**, superando a RandomForest y a la Regresión Logística.
- Se exportó el modelo entrenado, quedando listo para ser contenedorizado con Docker y desplegado en Kubernetes.